# W14-D2 概念实验：三源语义结构对账——谁是 Source of Truth？

配套阅读：`第14周-Day2-三源语义结构对账-谁是SourceOfTruth.md`（md 是对账与裁决，本 notebook 用**可执行实验**验证当天核心概念）：

1. **实验一 · 对账矩阵**：12 实施模块 ↔ 14 BCM 域 ↔ 17 Context 的名字级映射（今日手工校准 v0.1）——多对多扇出可视化，验证"三套切分逻辑没有显式映射就没有对账"；
2. **实验二 · capability 标签审计**：真实解析 `business-ontology.yaml` + 真实 `openspec/specs/` 273 目录——量化"贴纸 vs 锚点"（42% 正向匹配 / 12% 反向覆盖）；
3. **实验三 · 一个术语的四个坐标 + 语义路由器**："车位"在三源中的坐标漂移 + 分层 SoT 路由演示——验证"词汇认识对象，语义才认识事实"。

与明日 D3（实验1：ontology 文件内 schema/别名质量审计）的分工：今天看**跨源**一致性，明天看**源内**质量。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

## 实验一：对账矩阵——三套切分逻辑的多对多扇出

md §2.1 的手工映射表编码为数据。对账的核心事实：**扇出不是异常，是常态**——
- 资源管理 1 模块 → 2 Context（物理状态 vs 商业可用性，D-001 Amendment A）
- 财务管理 1 模块 → 4 Context（财务链拆成 Billing/Collection/Invoice/Accounting）
- 运营管理 ↔ 客服工单互相渗透（2→2）

本实验问：如果没有这张映射表，"铺位归属问题"拿 ontology 回答会命中哪个 Context？答案是 2 个——歧义本身就是断链。

In [ ]:
# W14-D2 手工校准的名字级映射 v0.1（依据：Domain Model §2 Coverage Matrix + Crosswalk + ontology 实测）
# 值 = 该模块的业务落到哪些 Context（编号沿用 Domain Model §2）
MODULE_TO_CONTEXTS = {
    "资源管理":       [1, 5],          # 01 Asset Foundation + 05 Lease/Occupancy（D-001 A 拆分）
    "招商管理":       [3],             # 03 Leasing Pipeline
    "合同管理":       [4],             # 04 Contract Lifecycle（BCM 已更名"协议管理"）
    "财务管理":       [6, 7, 8, 9],    # Billing/Collection/TaxInvoice/AccountingBridge 财务链四拆
    "运营管理":       [10, 13],        # 10 Operations + 13 Work Order（吞了客服的工单/投诉）
    "物业管理":       [11],
    "推广营销":       [14],
    "系统管理":       [],              # 横切平台层（用户/权限/流程），非业务域
    "资产管理(对接)": [],              # 外部系统集成，BCM/Context 均无对应
    "移动端":         [],              # 渠道不是域
    "数据决策":       [17],            # 17 BI & Analytics
    "预算管理":       [],              # 跨域挂靠 04 财务 + spec asset-budget-planning
}
CONTEXTS = {1:"Asset Foundation",2:"Merchant",3:"Leasing Pipeline",4:"Contract Lifecycle",
            5:"Lease/Occupancy",6:"Billing & AR",7:"Collection",8:"Tax Invoice",
            9:"Accounting Bridge",10:"Operations",11:"Property",12:"Engineering",
            13:"Work Order",14:"Marketing",15:"Customer/Member",16:"Parking",17:"BI & Analytics"}

fanout = {m: len(cs) for m, cs in MODULE_TO_CONTEXTS.items()}
multi  = {m: n for m, n in fanout.items() if n > 1}
orphan = [m for m, n in fanout.items() if n == 0]
covered = set(c for cs in MODULE_TO_CONTEXTS.values() for c in cs)
uncovered_ctx = [CONTEXTS[c] for c in sorted(CONTEXTS) if c not in covered]

print("== 对账摘要 ==")
print(f"模块总数 {len(MODULE_TO_CONTEXTS)} | 有 Context 落点 {len(MODULE_TO_CONTEXTS)-len(orphan)} | 无落点(横切/集成/渠道/挂靠) {len(orphan)}: {orphan}")
print(f"多对多扇出模块 {len(multi)} 个: {multi}")
print(f"ontology 模块层未覆盖的 Context {len(uncovered_ctx)}/17: {uncovered_ctx}")
print("注: Context 16 Parking 在模块层无入口，仅以 parking-space capability 挂在 资源管理/车位管理 子功能下")

In [ ]:
# 可视化：12 模块 × 17 Context 对账热力图
mods = list(MODULE_TO_CONTEXTS.keys())
M = np.zeros((len(mods), 17))
for i, m in enumerate(mods):
    for c in MODULE_TO_CONTEXTS[m]:
        M[i, c-1] = 1

fig, ax = plt.subplots(figsize=(13, 6))
ax.imshow(M, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1.2)
ax.set_xticks(range(17)); ax.set_xticklabels([f"{c:02d}\n{CONTEXTS[c].split(' /')[0]}" for c in range(1,18)], fontsize=8)
ax.set_yticks(range(len(mods))); ax.set_yticklabels(mods, fontsize=9)
for i in range(len(mods)):
    for j in range(17):
        if M[i, j]: ax.text(j, i, "●", ha="center", va="center", fontsize=11)
ax.set_title("W14-D2 对账矩阵 v0.1：实施模块 × Bounded Context（名字级手工映射）\n红点=业务落点；扇出行(资源管理1→2、财务管理1→4)与空行(横切/渠道)同时存在", fontsize=11)
plt.tight_layout(); plt.savefig("/root/learning-notebooks/第14周/w14d2_对账矩阵.png", dpi=110); plt.show()
print("结论：多对多是常态 → 任何'模块名≈Context名'的假设都会在 资源管理/财务管理 两个扇出行上翻车")

## 实验二：capability 标签审计——贴纸 vs 锚点（真实文件解析）

直接读 `/root/docs/lanlnk/config/ontology/business-ontology.yaml`（1572 行）和 `/root/lnkcre/openspec/specs/`（273 目录）。
验证 md §1.2② 的三个数字：**23 个 L 编号占位、命名标签仅 42% 能对上 spec、273 个 spec 只有 12% 被反向引用**。
对比锚点标准：BCM 的 CRE-* 行 ID 不可变 + Crosswalk 250 条显式映射——标签体系缺的正是"行 ID 锚点 + 显式映射"这两样。

In [ ]:
import yaml, os, re, collections

with open("/root/docs/lanlnk/config/ontology/business-ontology.yaml") as f:
    onto = yaml.safe_load(f)
specs = set(os.listdir("/root/lnkcre/openspec/specs/"))

cap_tags = collections.defaultdict(list)   # 标签 -> 挂载点(模块/子功能)
for m, md in onto["modules"].items():
    for s, sd in md.get("sub_functions", {}).items():
        for c in sd.get("capabilities", []):
            cap_tags[c].append(f"{m}/{s}")

is_placeholder = lambda c: re.fullmatch(r"L\d+", c) is not None
placeholders = {c for c in cap_tags if is_placeholder(c)}
named = {c for c in cap_tags if not is_placeholder(c)}
matched = {c for c in named if c in specs}
unmatched_named = named - matched
reuse = {c: v for c, v in cap_tags.items() if len(v) > 1 and not is_placeholder(c)}
top_reuse = sorted(reuse.items(), key=lambda x: -len(x[1]))[:3]

n_terms = sum(len(sd.get("terms", [])) for md in onto["modules"].values() for sd in md.get("sub_functions", {}).values())
print(f"实测：模块 {len(onto['modules'])} | 子功能 {sum(len(m.get('sub_functions',{})) for m in onto['modules'].values())} | 术语 {n_terms} | capability 标签 {len(cap_tags)}")
print(f"标签构成：L编号占位 {len(placeholders)} | 命名且匹配 spec {len(matched)} ({len(matched)/len(named)*100:.0f}%) | 命名但无 spec {len(unmatched_named)} ({len(unmatched_named)/len(named)*100:.0f}%)")
print(f"反向覆盖：{len(matched)}/{len(specs)} = {len(matched)/len(specs)*100:.0f}% 的 openspec spec 被 ontology 引用")
print(f"跨子功能复用标签 {len(reuse)} 个，TOP3：", [(c, len(v)) for c, v in top_reuse])
print(f"\n'万能贴纸'证据：platform-foundation 挂载 {len(cap_tags.get('platform-foundation',[]))} 个子功能，横跨 {len(set(x.split('/')[0] for x in cap_tags.get('platform-foundation',[])))} 个模块")

In [ ]:
# 可视化：标签构成 + 未被 ontology 覆盖的 spec 前缀 TOP8
unmatched_specs = specs - set(named)
pref = collections.Counter(s.split("-")[0] for s in unmatched_specs)
top8 = pref.most_common(8)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
a = axes[0]
vals = [len(placeholders), len(matched), len(unmatched_named)]
labels = [f"L编号占位\n{vals[0]}个\n(无语义)", f"命名+匹配spec\n{vals[1]}个\n(可作锚点)", f"命名+无spec\n{vals[2]}个\n(悬空标签)"]
colors = ["#bbbbbb", "#2e7d32", "#e65100"]
bars = a.bar(range(3), vals, color=colors)
a.set_xticks(range(3)); a.set_xticklabels(labels, fontsize=9)
for b, v in zip(bars, vals): a.text(b.get_x()+b.get_width()/2, v+0.5, str(v), ha="center", fontsize=10)
a.set_title(f"102 个 capability 标签的构成\n可作锚点的仅 {len(matched)}/{len(cap_tags)} = {len(matched)/len(cap_tags)*100:.0f}%", fontsize=10)

b = axes[1]
names = [p for p, _ in top8]; counts = [c for _, c in top8]
b.barh(range(len(names))[::-1], counts, color="#1565c0")
b.set_yticks(range(len(names))[::-1]); b.set_yticklabels(names, fontsize=9)
for i, c in enumerate(counts): b.text(c+0.3, len(names)-1-i, str(c), va="center", fontsize=9)
b.set_xlabel("spec 数量", fontsize=9)
b.set_title(f"未被 ontology 引用的 spec 前缀 TOP8（共 {len(unmatched_specs)} 个）\nanalytics/park/workflow...——实现已跑在词汇前面", fontsize=10)
plt.tight_layout(); plt.savefig("/root/learning-notebooks/第14周/w14d2_标签审计.png", dpi=110); plt.show()
print("解读：左图=正向审计(ontology 视角看标签质量)，右图=反向审计(spec 视角看词汇覆盖)")

## 实验三：一个术语的四个坐标 + 语义路由器

**Part A**：同一个词"车位"在三源中的坐标完全不同——这不是 bug，是分层设计的必然结果，但没有对账层就等于四个人在说四种语言。
**Part B**：分层 SoT 路由器。五个语义维度各有一个权威源；用 ontology 独答四个真实问题 → 三个断链；接线后全部命中。
断链时的正确行为是 **fail-closed 显式失败**（W10-D4 结论），而不是让 LLM 用语感编一个答案继续走（PT-W4-D6 的"无声失败"）。

In [ ]:
# Part A：'车位'的四个坐标（真实数据，今日对账实测）
drift = [
    ("business-ontology.yaml", "资源管理 → 车位管理(子功能)\ncapability 标签: parking-space\n术语: 车位/车位资料/车位信息"),
    ("CRE BCM", "11 停车管理域\n行 ID 前缀: CRE-PKG\n(published v0.3)"),
    ("MI Domain Model", "Context 16 Parking Management\n(P1, Boundary+Object 深度)"),
    ("openspec specs", "parking-space/ 目录\n(273 分之 1，实现真值)"),
]
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.axis("off")
xs = np.linspace(0.08, 0.92, 4)
for i, (src, coord) in enumerate(drift):
    ax.annotate("", xy=(xs[i]-0.09, 0.5), xytext=(xs[i]-0.035, 0.5),
                arrowprops=dict(arrowstyle="<->", lw=1.4, color="#888"))
    ax.add_patch(plt.Rectangle((xs[i]-0.115, 0.18), 0.23, 0.64, fill=False, ec="#1565c0", lw=1.6))
    ax.text(xs[i], 0.60, src, ha="center", fontsize=10, weight="bold")
    ax.text(xs[i], 0.36, coord, ha="center", fontsize=8.5, va="center")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("同一术语'车位'的四个坐标：四套坐标系都对，缺的是坐标换算层（= Semantic Model 的对账层）", fontsize=11)
plt.tight_layout(); plt.savefig("/root/learning-notebooks/第14周/w14d2_术语坐标漂移.png", dpi=110); plt.show()

In [ ]:
# Part B：分层 SoT 语义路由器（fail-closed 版）
AUTHORITIES = {
    "术语/别名":        ("business-ontology.yaml", "883 术语 + 模块别名", "A101/多经点位/车位 → 听得懂"),
    "业务能力/行ID":    ("CRE BCM", "CRE-* 行 ID 不可变(250条已Crosswalk)", "能力叫什么、归属哪个域"),
    "对象归属/生命周期": ("MI Domain Model", "17 Context + Object Ownership", "对象是谁的、状态怎么迁移"),
    "规则/Effect类型":  ("effect-registry.yaml", "5 类冻结(state-transition/occupancy/financial/lead-conversion/maintenance)", "生命周期事件传播什么"),
    "实现事实/验收":    ("openspec specs + 代码", "273 specs", "做没做、验收口径是什么"),
}
QUESTIONS = [  # (问题, 应路由的维度)
    ("'A101' 是铺位还是车位？", "术语/别名"),
    ("CRE-LEA-007 资源状态的对象 Owner 是谁？", "对象归属/生命周期"),
    ("合同终止时铺位占用变化走哪类 effect？", "规则/Effect类型"),
    ("车位管理的实现到什么程度、验收口径？", "实现事实/验收"),
]

def route(question, dim, wired=True):
    """wired=True: 分层路由; wired=False: 只有 ontology 硬答（演示断链）"""
    if wired:
        src, asset, answers = AUTHORITIES[dim]        # 命中唯一权威
        return f"✅ {question}\n   → {src}（{asset}）负责回答：{answers}"
    if dim == "术语/别名":
        return f"✅ {question}\n   → business-ontology.yaml 可以答（词汇层是它的独有资产）"
    return f"❌ {question}\n   → ontology 断链：它没有 {dim} 维度的构件（fail-closed，拒绝编造）"

print("== 场景1：只有 business-ontology.yaml（词汇层独答） ==")
for q, d in QUESTIONS: print(route(q, d, wired=False) + "\n")
print("== 场景2：四源接线后的分层路由（Semantic Model 的职责） ==")
for q, d in QUESTIONS: print(route(q, d, wired=True) + "\n")
print("结论：ontology 是必要的（唯一词汇权威），但永远不充分——Semantic Model = 分层路由 + 接线校验，不是第五份文档")

## 结论（回 Today's Question）

- **business-ontology.yaml 是词汇层**：883 术语独有价值 + 42% 可锚定标签 + 15 条溯源场景，但缺五类构件、无治理头、场景层 15/102——当不了 Semantic Model；
- **分层 SoT**：术语→ontology / 能力→BCM / 归属→Domain Model / 规则→effect-registry / 实现specs，每维度一个权威，其余单向消费（D-001 模式升维）；
- **Semantic Model = 对账层**：坐标换算（实验一扇出）+ 锚点审计（实验二 42%/12%）+ 路由接线（实验三 fail-closed）——W15 LnkChatBI 术语库消费的第一个检验就是它。

**明日 D3**：ontology 源内体检——schema 校验 / 别名冲突 / 场景 source 分布 / 程序化模块×Context 映射表（替代今天的手工 v0.1）。